# 005 · The calendar Module

> **Standard Library › Dates & Times** · Python 3.14

The `calendar` module answers questions *about* the calendar — how many days a month has, whether a year is a leap year, which weekday a month starts on — and can print month and year grids as text. It complements `datetime`, which represents individual dates.

**In this notebook you will learn:**

- `monthrange`, `isleap` and `leapdays` (vs `DateTime.DaysInMonth` / `IsLeapYear`)
- the weekday constants and the `Day` / `Month` enums
- `day_name` / `month_name` and friends
- printing text calendars, and changing the first day of the week
- `monthcalendar` for grid layouts, and recipes like "the last Friday of the month"

## .NET mental model

| .NET | Python `calendar` |
|---|---|
| `DateTime.DaysInMonth(y, m)` | `calendar.monthrange(y, m)[1]` |
| `DateTime.IsLeapYear(y)` | `calendar.isleap(y)` |
| `DayOfWeek.Friday` | `calendar.FRIDAY` (but Monday = 0!) |
| `CultureInfo.CurrentCulture.DateTimeFormat.GetDayName(...)` | `calendar.day_name[i]` |
| `DateTimeFormatInfo.MonthNames` | `calendar.month_name` |
| `DateTimeFormatInfo.FirstDayOfWeek` | `calendar.firstweekday()` / `setfirstweekday()` |
| — (no text calendar in the BCL) | `calendar.month()`, `TextCalendar`, `HTMLCalendar` |

⚠️ **Gotcha:** `System.Globalization.Calendar` in .NET is about *calendar systems* (Gregorian, Hijri, Japanese…). Python's `calendar` module always uses the proleptic Gregorian calendar; it is a utility module, not a calendar-system abstraction.

## Days in a month: `monthrange`

`calendar.monthrange(year, month)` returns a pair: the **weekday of the first day** and the **number of days** in the month.

```csharp
// C#
int days = DateTime.DaysInMonth(2026, 2);                    // 28
DayOfWeek first = new DateTime(2026, 2, 1).DayOfWeek;        // Sunday
```

In [ ]:
import calendar
from datetime import date, timedelta

first_weekday, days = calendar.monthrange(2026, 2)
print(first_weekday, days)                 # 6 28 — 6 is calendar.SUNDAY
print(calendar.monthrange(2028, 2))        # (calendar.TUESDAY, 29) — leap year

last_day = date(2026, 2, calendar.monthrange(2026, 2)[1])
print(last_day)                            # 2026-02-28 — "end of month"

**Tip:** another common "end of month" trick needs no `calendar` at all: go to the first day of the *next* month and step back one day.

In [ ]:
def end_of_month(d: date) -> date:
    first_of_next = (d.replace(day=28) + timedelta(days=4)).replace(day=1)
    return first_of_next - timedelta(days=1)

print(end_of_month(date(2026, 2, 10)), end_of_month(date(2028, 2, 10)), end_of_month(date(2026, 12, 31)))

## Leap years: `isleap` and `leapdays`

The Gregorian rule: divisible by 4, except centuries, except centuries divisible by 400. `leapdays(y1, y2)` counts leap years in the half-open range `[y1, y2)`.

In [ ]:
for year in (2024, 2026, 1900, 2000):
    print(year, calendar.isleap(year))    # C#: DateTime.IsLeapYear(year)

print(calendar.leapdays(2000, 2030))       # 8 — 2000, 2004, …, 2028

## Weekday constants and the `Day` / `Month` enums

The weekday constants follow `date.weekday()`: **Monday is 0** and Sunday is 6 — unlike .NET's `DayOfWeek` where Sunday is 0. Since Python 3.12 they are members of the `calendar.Day` `IntEnum`, and months are available as `calendar.Month` (January = 1).

In [ ]:
print(calendar.MONDAY, calendar.FRIDAY, calendar.SUNDAY)     # 0 4 6
print(repr(calendar.FRIDAY))                                 # an enum member
print(calendar.Day(4).name, calendar.Month(3).name)          # FRIDAY MARCH
print(calendar.FRIDAY == date(2026, 3, 6).weekday())         # True — 6 March 2026 is a Friday
print(calendar.weekday(2026, 3, 5))                          # 3 (Thursday), same as date.weekday()

## Names: `day_name`, `month_name` and abbreviations

These are sequences indexed like the constants. Note that `month_name` and `month_abbr` have an **empty string at index 0** so that `month_name[3]` is March:

In [ ]:
print(list(calendar.day_name))
print(list(calendar.day_abbr))
print(calendar.month_name[3], calendar.month_abbr[3])    # March Mar
print(repr(calendar.month_name[0]))                      # '' — placeholder
print(calendar.day_name[date(2026, 3, 5).weekday()])     # Thursday

⚠️ **Gotcha:** like `strftime("%A")`, the names come from the current **locale**, so they change if something calls `locale.setlocale` *(see Standard Library › Dates & Times › 002 · Formatting and Parsing Dates)*.

## Text calendars

`calendar.month(year, month)` returns a ready-made text grid, and `calendar.calendar(year)` returns a whole year. There is nothing like this in the .NET BCL — handy for CLIs and quick debugging.

In [ ]:
print(calendar.month(2026, 2))

Running `python -m calendar 2026 3` in a terminal prints the same thing. There is also an `HTMLCalendar` class that produces an HTML `<table>`:

In [ ]:
html = calendar.HTMLCalendar().formatmonth(2026, 3)
print(html.splitlines()[0])               # <table ... class="month">
print(html.count("<tr>"))                 # header rows + week rows

## First day of the week

By default weeks start on Monday (ISO 8601). In the US, weeks usually start on Sunday — the equivalent of `DateTimeFormatInfo.FirstDayOfWeek`. Prefer creating a `TextCalendar` with an explicit `firstweekday` over the **global** `setfirstweekday`:

In [ ]:
sunday_first = calendar.TextCalendar(firstweekday=calendar.SUNDAY)
print(sunday_first.formatmonth(2026, 3))

⚠️ **Gotcha:** `calendar.setfirstweekday()` changes module-level state used by `calendar.month()`, `monthcalendar()` and friends — for the whole process. If you must use it, restore the previous value:

In [ ]:
previous = calendar.firstweekday()
calendar.setfirstweekday(calendar.SUNDAY)
try:
    print(calendar.monthcalendar(2026, 3)[0])    # first week now starts on Sunday 1 March
finally:
    calendar.setfirstweekday(previous)

print(calendar.firstweekday())                   # 0 — back to Monday

## `monthcalendar` — a month as a grid

`monthcalendar(year, month)` returns a list of weeks; each week is a list of 7 day numbers, with **0** for days that belong to the neighbouring months. It is perfect for rendering a calendar UI.

In [ ]:
for week in calendar.monthcalendar(2026, 3):
    print(week)

If you want real `date` objects (including the neighbouring months' days), use a `calendar.Calendar` instance:

In [ ]:
cal = calendar.Calendar(firstweekday=calendar.MONDAY)
weeks = cal.monthdatescalendar(2026, 3)            # list of weeks of date objects
print(len(weeks), weeks[0][0], weeks[-1][-1])      # 6 weeks, from Mon 23 Feb to Sun 5 Apr

march_days = [d for d in cal.itermonthdates(2026, 3) if d.month == 3]
print(len(march_days))                              # 31

## Recipe: "the last Friday of the month"

Business rules like "pay day is the last Friday" or "maintenance on the second Tuesday" are easy with `monthcalendar`: take the weekday's column and skip the zeros.

```csharp
// C# — typical manual approach
var last = new DateTime(2026, 3, DateTime.DaysInMonth(2026, 3));
while (last.DayOfWeek != DayOfWeek.Friday) last = last.AddDays(-1);
```

In [ ]:
def nth_weekday(year: int, month: int, weekday: int, n: int) -> date:
    """n-th weekday of the month; n=1 is the first, n=-1 the last."""
    days = [week[weekday] for week in calendar.monthcalendar(year, month) if week[weekday] != 0]
    return date(year, month, days[n - 1 if n > 0 else n])

print(nth_weekday(2026, 3, calendar.FRIDAY, -1))     # 2026-03-27 — last Friday
print(nth_weekday(2026, 3, calendar.TUESDAY, 2))     # 2026-03-10 — second Tuesday
print(nth_weekday(2026, 11, calendar.THURSDAY, 4))   # 2026-11-26 — US Thanksgiving

The same result with plain arithmetic, starting from the last day and stepping back (the Python version of the C# loop, without the loop):

In [ ]:
def last_weekday(year: int, month: int, weekday: int) -> date:
    last = date(year, month, calendar.monthrange(year, month)[1])
    return last - timedelta(days=(last.weekday() - weekday) % 7)

print(last_weekday(2026, 3, calendar.FRIDAY))                                  # 2026-03-27
print([str(last_weekday(2026, m, calendar.FRIDAY)) for m in range(1, 4)])

## `calendar.timegm` — UTC struct to Unix timestamp

One odd but useful extra: `calendar.timegm` is the inverse of `time.gmtime`, turning a UTC time tuple into a Unix timestamp. With aware datetimes you would normally just call `.timestamp()`.

In [ ]:
from datetime import datetime, timezone

print(calendar.timegm((2026, 3, 5, 14, 30, 15)))                           # 1772721015
print(int(datetime(2026, 3, 5, 14, 30, 15, tzinfo=timezone.utc).timestamp()))   # same

## Summary

| Task | C# / .NET | Python |
|---|---|---|
| Days in month | `DateTime.DaysInMonth(y, m)` | `calendar.monthrange(y, m)[1]` |
| First weekday of month | `new DateTime(y, m, 1).DayOfWeek` | `calendar.monthrange(y, m)[0]` |
| Leap year? | `DateTime.IsLeapYear(y)` | `calendar.isleap(y)` |
| Count leap years | loop | `calendar.leapdays(y1, y2)` |
| Weekday constant | `DayOfWeek.Friday` (= 5) | `calendar.FRIDAY` (= 4) |
| Day / month names | `DateTimeFormatInfo.DayNames` / `MonthNames` | `calendar.day_name` / `calendar.month_name` |
| First day of week | `DateTimeFormatInfo.FirstDayOfWeek` | `TextCalendar(firstweekday=...)` |
| Month grid | — | `calendar.monthcalendar(y, m)` |
| Text calendar | — | `calendar.month(y, m)` |
| Last Friday | loop with `AddDays(-1)` | `monthcalendar` column or `monthrange` arithmetic |

## Exercises

1. Print every month of 2026 that has **five** Fridays (use `monthcalendar`).
2. Write `days_in_year(year) -> int` using `calendar` and check it for 2026, 2028 and 2100.
3. Many companies close the books on the **last business day** of the month. Write `last_business_day(year, month) -> date` (Monday–Friday, ignore holidays) and print it for every month of 2026.

In [ ]:
# Your solutions here

<details>
<summary>Show solutions</summary>

```python
import calendar
from datetime import date, timedelta

# 1 — a month has five Fridays if five weeks have a non-zero Friday column
for month in range(1, 13):
    fridays = [week[calendar.FRIDAY] for week in calendar.monthcalendar(2026, month)
               if week[calendar.FRIDAY]]
    if len(fridays) == 5:
        print(calendar.month_name[month], fridays)

# 2 — isleap returns a bool, and True == 1
def days_in_year(year: int) -> int:
    return 365 + calendar.isleap(year)

print([days_in_year(y) for y in (2026, 2028, 2100)])   # [365, 366, 365]

# 3 — start at the last day and step back over Saturday (5) / Sunday (6)
def last_business_day(year: int, month: int) -> date:
    d = date(year, month, calendar.monthrange(year, month)[1])
    while d.weekday() >= calendar.SATURDAY:
        d -= timedelta(days=1)
    return d

for month in range(1, 13):
    print(last_business_day(2026, month).strftime("%a %d %b"))
```

</details>